
# Single-Topic Forgetting-Kernel Study
## Experimental design + full simulation notebook

This notebook redesigns the project around the **single-topic setting**, which is the cleanest way to answer the main research question:

\[
\textbf{How does a forgetting kernel change the dynamics of the baseline model under different networks } W \textbf{ and initial conditions } X_0 \textbf{?}
\]

The notebook is organized in **two stages**.

## Stage 1 — How should impact be measured?

We compare the baseline DeGroot model with a forgetting-kernel extension under different:

- network structures \(W\)
- initial conditions \(X_0\)
- forgetting parameters \((\lambda, H)\)

and measure the effect using several metrics:

- **Path Gap**
- **Gap AUC**
- **Final Gap**
- **Average Disagreement Change**
- **Final Disagreement Change**
- **Convergence-Time Change**

This stage is about defining a **scientifically meaningful and robust notion of impact**.

## Stage 2 — How can the observed effects be explained mathematically?

We connect the observed numerical behavior to mathematical structure by:

- constructing the **memory-augmented dynamics**
- building the **augmented system matrix**
- comparing its spectral properties with the baseline matrix
- relating network structure and memory length to convergence behavior

This stage is about explaining **why** the forgetting kernel changes the dynamics.

## Models

### Baseline DeGroot model
\[
x(t+1) = W x(t)
\]

### Forgetting-kernel model
\[
x(t+1) = W \tilde{x}(t), \qquad \tilde{x}(t)=\sum_{\tau=0}^{H}K(\tau)\,x(t-\tau)
\]

For an exponential forgetting kernel:
\[
K(\tau)\propto e^{-\lambda \tau}
\]

## Notebook outputs

This notebook saves all result tables and figures to the **current working directory**.


In [ ]:

import os
from collections import deque
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from scipy import sparse
from IPython.display import display

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
np.set_printoptions(precision=4, suppress=True)

SAVE_DIR = os.getcwd()
print("Current working directory:", SAVE_DIR)
print("All CSV files and figures will be saved here.")


In [ ]:

def savefig_local(filename: str, dpi: int = 220):
    path = os.path.join(SAVE_DIR, filename)
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print("Saved figure:", path)



## Experimental design

The factorial structure of the experiments is:

\[
(W\text{-type}, X_0\text{-type}, \lambda, H) \longrightarrow \text{dynamical metrics}
\]

### Networks \(W\)
We include several representative network families:

- **Sparse Watts–Strogatz**
- **Dense Watts–Strogatz**
- **Erdős–Rényi**
- **Barabási–Albert**

### Initial conditions \(X_0\)
We include several representative single-topic initial states:

- **normal**: moderate disagreement around zero
- **bimodal**: polarized population
- **skewed**: asymmetric clusters

### Kernel parameters
We tune:

- \(\lambda\): forgetting rate
- \(H\): memory horizon

## Important implementation decision

For the forgetting model, this notebook uses the **exact finite-horizon exponential kernel**:

\[
K(\tau)=\frac{e^{-\lambda\tau}}{\sum_{s=0}^{H} e^{-\lambda s}}, \qquad \tau=0,\dots,H
\]

This is more faithful to the intended model than a pure recursive approximation, while still remaining computationally fast enough for the planned experiments.


In [ ]:

CONFIG = {
    "n_agents": 100,
    "T": 120,

    "opinion_low": -1.0,
    "opinion_high": 1.0,

    "network_families": ["ws_sparse", "ws_dense", "er", "ba"],
    "x0_families": ["normal", "bimodal", "skewed"],

    "ws_sparse_k": 4,
    "ws_dense_k": 10,
    "ws_p": 0.08,
    "er_p": 0.06,
    "ba_m": 3,
    "self_weight": 0.15,

    "normal_std": 0.35,
    "bimodal_mean": 0.5,
    "bimodal_std": 0.18,
    "skewed_major_mean": 0.25,
    "skewed_minor_mean": -0.7,
    "skewed_major_std": 0.15,
    "skewed_minor_std": 0.12,
    "skewed_major_prob": 0.8,

    "lambda_grid_coarse": np.array([0.001, 0.003, 0.006, 0.01, 0.02, 0.04, 0.08]),
    "H_grid_coarse": np.array([4, 8, 12, 20, 32, 48, 64]),

    "lambda_refine_factors": np.array([0.5, 0.75, 1.0, 1.25, 1.5]),
    "H_refine_offsets": np.array([-12, -8, -4, 0, 4, 8, 12]),

    "conv_eps": 1e-4,
    "seed": 42,
    "top_k": 10,
}

rng = np.random.default_rng(CONFIG["seed"])


## Utility functions

In [ ]:

def minmax_scale(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=float)
    mn, mx = arr.min(), arr.max()
    if np.isclose(mx, mn):
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

def row_normalize_sparse(A: sparse.csr_matrix) -> sparse.csr_matrix:
    row_sums = np.asarray(A.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0
    inv_row = sparse.diags(1.0 / row_sums)
    return (inv_row @ A).tocsr()

def convergence_time(traj: np.ndarray, eps: float = 1e-4) -> int:
    diffs = np.linalg.norm(np.diff(traj, axis=0), axis=1)
    idx = np.where(diffs < eps)[0]
    if len(idx) == 0:
        return len(traj) - 1
    return int(idx[0] + 1)

def plot_heatmap(df: pd.DataFrame, value_col: str, title: str, filename: str = None):
    pivot = df.pivot(index="H", columns="lambda", values=value_col)
    plt.figure(figsize=(9, 5))
    plt.imshow(pivot.values, aspect="auto", origin="lower")
    plt.xticks(range(len(pivot.columns)), [f"{x:.4f}" for x in pivot.columns], rotation=45)
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.xlabel("lambda")
    plt.ylabel("H")
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    if filename is not None:
        savefig_local(filename)
    plt.show()


## Network generators

In [ ]:

def generate_network_matrix(family: str, n_agents: int, self_weight: float, seed: int) -> sparse.csr_matrix:
    if family == "ws_sparse":
        G = nx.watts_strogatz_graph(n=n_agents, k=CONFIG["ws_sparse_k"], p=CONFIG["ws_p"], seed=seed)
    elif family == "ws_dense":
        G = nx.watts_strogatz_graph(n=n_agents, k=CONFIG["ws_dense_k"], p=CONFIG["ws_p"], seed=seed)
    elif family == "er":
        G = nx.erdos_renyi_graph(n=n_agents, p=CONFIG["er_p"], seed=seed)
    elif family == "ba":
        G = nx.barabasi_albert_graph(n=n_agents, m=CONFIG["ba_m"], seed=seed)
    else:
        raise ValueError(f"Unknown network family: {family}")

    A = nx.to_scipy_sparse_array(G, format="csr", dtype=float)
    A = A + sparse.eye(n_agents, format="csr") * self_weight
    return row_normalize_sparse(A)


## Initial-condition generators

In [ ]:

def generate_initial_condition(family: str, n_agents: int, low: float, high: float, seed: int) -> np.ndarray:
    rng_local = np.random.default_rng(seed)

    if family == "normal":
        x0 = rng_local.normal(0.0, CONFIG["normal_std"], n_agents)
    elif family == "bimodal":
        mask = rng_local.random(n_agents) < 0.5
        x0 = np.where(
            mask,
            rng_local.normal(-CONFIG["bimodal_mean"], CONFIG["bimodal_std"], n_agents),
            rng_local.normal(CONFIG["bimodal_mean"], CONFIG["bimodal_std"], n_agents)
        )
    elif family == "skewed":
        mask = rng_local.random(n_agents) < CONFIG["skewed_major_prob"]
        x0 = np.where(
            mask,
            rng_local.normal(CONFIG["skewed_major_mean"], CONFIG["skewed_major_std"], n_agents),
            rng_local.normal(CONFIG["skewed_minor_mean"], CONFIG["skewed_minor_std"], n_agents)
        )
    else:
        raise ValueError(f"Unknown initial-condition family: {family}")

    return np.clip(x0, low, high)


## Forgetting kernel

In [ ]:

def exponential_kernel_exact(lam: float, H: int) -> np.ndarray:
    taus = np.arange(H + 1, dtype=float)
    raw = np.exp(-lam * taus)
    return raw / raw.sum()


## Simulation engines

In [ ]:

def simulate_degroot(x0: np.ndarray, W: sparse.csr_matrix, T: int) -> np.ndarray:
    n_agents = len(x0)
    traj = np.zeros((T + 1, n_agents), dtype=float)
    traj[0] = x0.copy()

    x = x0.copy()
    for t in range(T):
        x = W.dot(x)
        traj[t + 1] = x

    return traj

def simulate_forgetting_exact(x0: np.ndarray, W: sparse.csr_matrix, T: int, lam: float, H: int) -> np.ndarray:
    n_agents = len(x0)
    traj = np.zeros((T + 1, n_agents), dtype=float)
    traj[0] = x0.copy()

    kernel = exponential_kernel_exact(lam, H)
    x = x0.copy()
    history = deque([x0.copy()], maxlen=H + 1)

    for t in range(T):
        current_history = list(history)
        weights = kernel[:len(current_history)]
        mem = np.zeros_like(x)
        for w, state in zip(weights, current_history):
            mem += w * state
        x = W.dot(mem)
        traj[t + 1] = x
        history.appendleft(x.copy())

    return traj


## Metrics

In [ ]:

def disagreement_over_time(traj: np.ndarray) -> np.ndarray:
    return traj.var(axis=1)

def path_gap_series(traj_base: np.ndarray, traj_forget: np.ndarray) -> np.ndarray:
    return np.linalg.norm(traj_forget - traj_base, axis=1)

def collect_metrics(traj_base: np.ndarray, traj_forget: np.ndarray, conv_eps: float) -> dict:
    gap_series = path_gap_series(traj_base, traj_forget)
    d_base = disagreement_over_time(traj_base)
    d_forget = disagreement_over_time(traj_forget)

    conv_base = convergence_time(traj_base, conv_eps)
    conv_forget = convergence_time(traj_forget, conv_eps)

    return {
        "path_gap": float(gap_series.mean()),
        "gap_auc": float(gap_series.sum()),
        "final_gap": float(gap_series[-1]),
        "avg_disagreement_base": float(d_base.mean()),
        "avg_disagreement_forget": float(d_forget.mean()),
        "delta_avg_disagreement": float(d_forget.mean() - d_base.mean()),
        "final_disagreement_base": float(d_base[-1]),
        "final_disagreement_forget": float(d_forget[-1]),
        "delta_final_disagreement": float(d_forget[-1] - d_base[-1]),
        "conv_time_base": int(conv_base),
        "conv_time_forget": int(conv_forget),
        "delta_conv_time": int(conv_forget - conv_base),
    }


## Scenario builder

In [ ]:

def build_scenario(network_family: str, x0_family: str, seed: int):
    W = generate_network_matrix(family=network_family, n_agents=CONFIG["n_agents"], self_weight=CONFIG["self_weight"], seed=seed)
    x0 = generate_initial_condition(family=x0_family, n_agents=CONFIG["n_agents"], low=CONFIG["opinion_low"], high=CONFIG["opinion_high"], seed=seed)
    traj_base = simulate_degroot(x0=x0, W=W, T=CONFIG["T"])
    return W, x0, traj_base


## Coarse-to-fine tuning

In [ ]:

def add_simple_impact_score(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["path_gap_norm"] = minmax_scale(out["path_gap"].values)
    out["delta_avg_disagreement_norm"] = minmax_scale(out["delta_avg_disagreement"].values)
    out["impact_score"] = 0.5 * out["path_gap_norm"] + 0.5 * out["delta_avg_disagreement_norm"]
    return out

def run_grid_search(W, x0, traj_base, lambda_grid, H_grid, network_family, x0_family) -> pd.DataFrame:
    rows = []
    for lam in lambda_grid:
        for H in H_grid:
            traj_forget = simulate_forgetting_exact(x0=x0, W=W, T=CONFIG["T"], lam=float(lam), H=int(H))
            metrics = collect_metrics(traj_base=traj_base, traj_forget=traj_forget, conv_eps=CONFIG["conv_eps"])
            rows.append({
                "network_family": network_family,
                "x0_family": x0_family,
                "lambda": float(lam),
                "H": int(H),
                **metrics
            })
    df = pd.DataFrame(rows)
    df = add_simple_impact_score(df)
    return df.sort_values("impact_score", ascending=False).reset_index(drop=True)


## Example scenario visual check

In [ ]:

EXAMPLE_NETWORK = "ws_sparse"
EXAMPLE_X0 = "bimodal"

W_ex, x0_ex, traj_base_ex = build_scenario(network_family=EXAMPLE_NETWORK, x0_family=EXAMPLE_X0, seed=CONFIG["seed"])

plt.figure(figsize=(8, 4))
plt.hist(x0_ex, bins=24, alpha=0.8)
plt.title(f"Initial opinions: {EXAMPLE_X0}")
plt.xlabel("Opinion")
plt.ylabel("Count")
plt.tight_layout()
savefig_local("fig_example_initial_histogram.png")
plt.show()

plt.figure(figsize=(6, 5))
plt.imshow(W_ex.toarray(), aspect="auto")
plt.title(f"Influence matrix W: {EXAMPLE_NETWORK}")
plt.colorbar()
plt.tight_layout()
savefig_local("fig_example_W_heatmap.png")
plt.show()


## Full Stage 1 experiment

In [ ]:

all_coarse_tables = []
scenario_best_rows = []

for network_family, x0_family in product(CONFIG["network_families"], CONFIG["x0_families"]):
    W_sc, x0_sc, traj_base_sc = build_scenario(network_family=network_family, x0_family=x0_family, seed=CONFIG["seed"])
    df_sc = run_grid_search(
        W=W_sc,
        x0=x0_sc,
        traj_base=traj_base_sc,
        lambda_grid=CONFIG["lambda_grid_coarse"],
        H_grid=CONFIG["H_grid_coarse"],
        network_family=network_family,
        x0_family=x0_family
    )
    all_coarse_tables.append(df_sc)
    scenario_best_rows.append(df_sc.iloc[0].to_dict())

df_coarse_all = pd.concat(all_coarse_tables, ignore_index=True)
df_scenario_best = pd.DataFrame(scenario_best_rows).sort_values("impact_score", ascending=False).reset_index(drop=True)

display(df_scenario_best.head(CONFIG["top_k"]))


In [ ]:

coarse_all_path = os.path.join(SAVE_DIR, "single_topic_stage1_coarse_all.csv")
coarse_best_path = os.path.join(SAVE_DIR, "single_topic_stage1_coarse_best_by_scenario.csv")
df_coarse_all.to_csv(coarse_all_path, index=False)
df_scenario_best.to_csv(coarse_best_path, index=False)
print(coarse_all_path)
print(coarse_best_path)


## Metric relationships

In [ ]:

metric_cols = ["path_gap", "gap_auc", "final_gap", "delta_avg_disagreement", "delta_final_disagreement", "delta_conv_time"]
corr_table = df_coarse_all[metric_cols].corr(method="spearman")
display(corr_table)

plt.figure(figsize=(7, 6))
plt.imshow(corr_table.values, aspect="auto")
plt.xticks(range(len(metric_cols)), metric_cols, rotation=45, ha="right")
plt.yticks(range(len(metric_cols)), metric_cols)
plt.title("Spearman correlation among impact metrics")
plt.colorbar()
plt.tight_layout()
savefig_local("fig_metric_correlation_heatmap.png")
plt.show()


## Fine search for the strongest scenario

In [ ]:

strongest = df_scenario_best.iloc[0]
best_network = strongest["network_family"]
best_x0 = strongest["x0_family"]
lam_star = float(strongest["lambda"])
H_star = int(strongest["H"])

W_best, x0_best, traj_base_best = build_scenario(network_family=best_network, x0_family=best_x0, seed=CONFIG["seed"])
lambda_grid_fine = np.unique(np.clip(lam_star * CONFIG["lambda_refine_factors"], 1e-4, None))
H_grid_fine = np.unique(np.clip(H_star + CONFIG["H_refine_offsets"], 2, None).astype(int))

df_fine_best = run_grid_search(
    W=W_best,
    x0=x0_best,
    traj_base=traj_base_best,
    lambda_grid=lambda_grid_fine,
    H_grid=H_grid_fine,
    network_family=best_network,
    x0_family=best_x0
)

display(df_fine_best.head(CONFIG["top_k"]))


In [ ]:

fine_best_path = os.path.join(SAVE_DIR, "single_topic_stage1_fine_best_scenario.csv")
df_fine_best.to_csv(fine_best_path, index=False)
print(fine_best_path)


## Heatmaps for the strongest scenario

In [ ]:

prefix = f"{best_network}_{best_x0}"
plot_heatmap(df_fine_best, "impact_score", f"Fine search impact score ({best_network}, {best_x0})", f"fig_fine_impact_{prefix}.png")
plot_heatmap(df_fine_best, "path_gap", f"Fine search path gap ({best_network}, {best_x0})", f"fig_fine_path_gap_{prefix}.png")
plot_heatmap(df_fine_best, "delta_avg_disagreement", f"Fine search delta average disagreement ({best_network}, {best_x0})", f"fig_fine_delta_avg_disagreement_{prefix}.png")
plot_heatmap(df_fine_best, "delta_conv_time", f"Fine search delta convergence time ({best_network}, {best_x0})", f"fig_fine_delta_conv_time_{prefix}.png")


## Best forgetting trajectory for the strongest scenario

In [ ]:

best_fine_row = df_fine_best.iloc[0]
best_lambda = float(best_fine_row["lambda"])
best_H = int(best_fine_row["H"])
traj_forget_best = simulate_forgetting_exact(x0=x0_best, W=W_best, T=CONFIG["T"], lam=best_lambda, H=best_H)
gap_series_best = path_gap_series(traj_base_best, traj_forget_best)
d_base_best = disagreement_over_time(traj_base_best)
d_forget_best = disagreement_over_time(traj_forget_best)

plt.figure(figsize=(8, 4.5))
plt.plot(traj_base_best.mean(axis=1), label="Baseline mean")
plt.plot(traj_forget_best.mean(axis=1), label="Forgetting mean")
plt.title("Mean opinion over time")
plt.xlabel("Time")
plt.ylabel("Mean opinion")
plt.legend()
plt.tight_layout()
savefig_local("fig_best_mean_opinion.png")
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(d_base_best, label="Baseline disagreement")
plt.plot(d_forget_best, label="Forgetting disagreement")
plt.title("Disagreement over time")
plt.xlabel("Time")
plt.ylabel("Variance across agents")
plt.legend()
plt.tight_layout()
savefig_local("fig_best_disagreement.png")
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(gap_series_best)
plt.title("Trajectory gap over time")
plt.xlabel("Time")
plt.ylabel("Euclidean gap")
plt.tight_layout()
savefig_local("fig_best_trajectory_gap.png")
plt.show()



# Stage 2 — Mathematical explanation

The forgetting model is no longer a first-order system in \(x(t)\).  
Instead, it is a **memory-augmented dynamical system**.

If we define the augmented state
\[
z(t)=
\begin{bmatrix}
x(t)\\
x(t-1)\\
\vdots\\
x(t-H)
\end{bmatrix},
\]
then the forgetting model can be written as
\[
z(t+1)=A_{\text{aug}} z(t)
\]
for an augmented matrix \(A_{\text{aug}}\) built from:

- the baseline influence matrix \(W\)
- the kernel weights \(K(0),\dots,K(H)\)

This makes it possible to compare the spectral structure of:

- the baseline matrix \(W\)
- the memory-augmented matrix \(A_{\text{aug}}\)

and use that comparison to explain why trajectories, disagreement, and convergence change.


In [ ]:

def build_augmented_matrix(W: sparse.csr_matrix, lam: float, H: int) -> np.ndarray:
    Wd = W.toarray()
    n = Wd.shape[0]
    kernel = exponential_kernel_exact(lam, H)
    dim = n * (H + 1)
    A_aug = np.zeros((dim, dim), dtype=float)
    for j in range(H + 1):
        A_aug[:n, j*n:(j+1)*n] = kernel[j] * Wd
    for j in range(1, H + 1):
        A_aug[j*n:(j+1)*n, (j-1)*n:j*n] = np.eye(n)
    return A_aug


## Spectral comparison

In [ ]:

A_aug_best = build_augmented_matrix(W_best, best_lambda, best_H)
eig_W = np.linalg.eigvals(W_best.toarray())
abs_eig_W = np.sort(np.abs(eig_W))[::-1]
eig_A = np.linalg.eigvals(A_aug_best)
abs_eig_A = np.sort(np.abs(eig_A))[::-1]

spec_summary = pd.DataFrame({
    "baseline_top_abs_eigs": pd.Series(abs_eig_W[:10]),
    "augmented_top_abs_eigs": pd.Series(abs_eig_A[:10]),
})
display(spec_summary)

k_show = min(20, len(abs_eig_W), len(abs_eig_A))
plt.figure(figsize=(8, 4.5))
plt.plot(range(1, k_show + 1), abs_eig_W[:k_show], marker="o", label="Baseline W")
plt.plot(range(1, k_show + 1), abs_eig_A[:k_show], marker="o", label="Augmented A_aug")
plt.title("Absolute eigenvalues: baseline vs augmented system")
plt.xlabel("Rank")
plt.ylabel("Absolute eigenvalue")
plt.legend()
plt.tight_layout()
savefig_local("fig_spectral_comparison.png")
plt.show()


## Compact summary table

In [ ]:

summary_table = pd.DataFrame([{
    "best_network_family": best_network,
    "best_x0_family": best_x0,
    "best_lambda": best_lambda,
    "best_H": best_H,
    "path_gap": best_fine_row["path_gap"],
    "gap_auc": best_fine_row["gap_auc"],
    "final_gap": best_fine_row["final_gap"],
    "delta_avg_disagreement": best_fine_row["delta_avg_disagreement"],
    "delta_final_disagreement": best_fine_row["delta_final_disagreement"],
    "delta_conv_time": best_fine_row["delta_conv_time"],
    "impact_score": best_fine_row["impact_score"],
    "baseline_second_abs_eig": abs_eig_W[1] if len(abs_eig_W) > 1 else np.nan,
    "augmented_second_abs_eig": abs_eig_A[1] if len(abs_eig_A) > 1 else np.nan,
}])
display(summary_table)

summary_path = os.path.join(SAVE_DIR, "single_topic_stage2_summary.csv")
summary_table.to_csv(summary_path, index=False)
print(summary_path)
